# An honest comparison: decision tree from scratch vs scikit-learn

This notebook evaluates the educational NumPy implementation against `sklearn.tree.DecisionTreeClassifier` without selecting a split or result that makes the models appear more similar. It reports repeated measurements, paired differences, model structure, and API differences. Similar results on Iris are evidence about this experiment only; they are not evidence that the implementations are equivalent or that either model is generally better.

## What is and is not being compared

Both models greedily search numeric thresholds using Gini impurity. They are not the same algorithm in every operational detail:

- The custom model grows breadth-first and exposes `min_leaf_size`, `max_depth`, and `max_nodes`.
- scikit-learn uses an optimized production implementation with its own split-search and tie-breaking behavior, input validation, and additional stopping controls.
- The custom model's `min_leaf_size` is a node-level permission to split; it is not equivalent to scikit-learn's `min_samples_leaf`, which constrains both children.
- The custom model has no `random_state` parameter and its current majority-vote helper uses Python's global random generator for ties. The notebook therefore seeds that generator with the split seed before each custom fit/prediction; this makes the protocol repeatable but is not a model-level random-state feature. scikit-learn receives the same split seed through its own `random_state` parameter.
- The custom `predict_proba` returns per-leaf dictionaries, while scikit-learn returns a dense array aligned to `classes_`; these outputs are not directly interchangeable.

Because there is no perfectly equivalent hyperparameter mapping, the notebook reports two experiments instead of hiding the mismatch:

1. **Native defaults:** each model is used with its own default training behavior. This is the fairest comparison of the APIs as implemented, but not a claim of identical settings.
2. **Shared depth limit:** both models are constrained to the same numeric `max_depth` for depths 1 through 5. This isolates one common control while still reporting that other implementation differences remain.

In [ ]:
from pathlib import Path
import random
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier as SklearnDecisionTreeClassifier

from decision_tree import DecisionTreeClassifier

RANDOM_SEEDS = range(30)
TEST_SIZE = 0.20
iris = load_iris()
X, y = iris.data, iris.target
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} numeric features, {len(np.unique(y))} classes")
print(f"Each repetition: stratified {1 - TEST_SIZE:.0%}/{TEST_SIZE:.0%} split; seeds: 0 through 29")

## Experiment 1: native/default behavior across 30 holdouts

For every seed, both models receive exactly the same training and test rows. The custom model is fitted with its `fit` defaults; scikit-learn is fitted with its defaults except for `random_state`. The notebook also seeds Python's global random generator before each custom run because the current custom tie-breaking helper depends on it. No seed is selected after inspecting the results.

In [ ]:
native_rows = []
native_predictions = {}

for seed in RANDOM_SEEDS:
    X_train, X_test, y_train, y_test, train_ids, test_ids = train_test_split(
        X,
        y,
        np.arange(len(y)),
        test_size=TEST_SIZE,
        stratify=y,
        random_state=seed,
    )

    random.seed(seed)
    custom = DecisionTreeClassifier()
    custom.fit(X_train, y_train)
    sklearn_tree = SklearnDecisionTreeClassifier(random_state=seed)
    sklearn_tree.fit(X_train, y_train)

    custom_pred = custom.predict(X_test).astype(int)
    sklearn_pred = sklearn_tree.predict(X_test).astype(int)
    native_predictions[seed] = {
        "y_test": y_test,
        "test_ids": test_ids,
        "custom_pred": custom_pred,
        "sklearn_pred": sklearn_pred,
    }
    native_rows.extend(
        [
            {
                "seed": seed,
                "model": "custom",
                "accuracy": accuracy_score(y_test, custom_pred),
                "depth": custom.root.depth(),
                "nodes": custom.root.count_nodes(),
            },
            {
                "seed": seed,
                "model": "scikit-learn",
                "accuracy": accuracy_score(y_test, sklearn_pred),
                "depth": sklearn_tree.get_depth(),
                "nodes": sklearn_tree.tree_.node_count,
            },
        ]
    )

native_results = pd.DataFrame(native_rows)
native_summary = (
    native_results.groupby("model")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        accuracy_min=("accuracy", "min"),
        accuracy_max=("accuracy", "max"),
        depth_mean=("depth", "mean"),
        depth_min=("depth", "min"),
        depth_max=("depth", "max"),
        nodes_mean=("nodes", "mean"),
        nodes_min=("nodes", "min"),
        nodes_max=("nodes", "max"),
    )
    .reset_index()
)
display(native_summary.round(4))

In [ ]:
native_pair_rows = []
for seed, values in native_predictions.items():
    custom_pred = values["custom_pred"]
    sklearn_pred = values["sklearn_pred"]
    native_pair_rows.append(
        {
            "seed": seed,
            "custom_accuracy": accuracy_score(values["y_test"], custom_pred),
            "sklearn_accuracy": accuracy_score(values["y_test"], sklearn_pred),
            "custom_minus_sklearn_accuracy": accuracy_score(values["y_test"], custom_pred)
            - accuracy_score(values["y_test"], sklearn_pred),
            "prediction_agreement": np.mean(custom_pred == sklearn_pred),
            "different_predictions": np.sum(custom_pred != sklearn_pred),
        }
    )

native_pairs = pd.DataFrame(native_pair_rows)
print("Paired results across the same 30 test splits:")
display(native_pairs.round(4))
print(
    f"Mean accuracy difference (custom - scikit-learn): "
    f"{native_pairs['custom_minus_sklearn_accuracy'].mean():+.4f}"
)
print(
    f"Mean prediction agreement: {native_pairs['prediction_agreement'].mean():.2%}; "
    f"range: {native_pairs['prediction_agreement'].min():.2%} to "
    f"{native_pairs['prediction_agreement'].max():.2%}"
)

### Reading the native-default results

The mean is not a win condition. Report the spread and paired differences as well: a small mean difference can coexist with meaningful disagreement on particular splits. Accuracy is a test-set estimate, not a property of an implementation in isolation, and this experiment uses only one small dataset.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for model, color in [("custom", "tab:blue"), ("scikit-learn", "tab:orange")]:
    values = native_results[native_results["model"] == model]
    axes[0].plot(values["seed"], values["accuracy"], "o-", label=model, color=color)
    axes[1].plot(values["seed"], values["depth"], "o-", label=model, color=color)
    axes[2].plot(values["seed"], values["nodes"], "o-", label=model, color=color)
for ax, title, ylabel in zip(
    axes,
    ["Accuracy by split", "Learned depth by split", "Node count by split"],
    ["test accuracy", "depth", "nodes"],
):
    ax.set_title(title)
    ax.set_xlabel("split seed")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Experiment 2: isolate a shared `max_depth` control

This is not a claim that the models become equivalent. It answers a narrower question: when both are asked to stop at the same maximum depth, how do their measured outcomes compare over the same 30 splits? The custom depth is passed explicitly to `fit`, because this implementation's `fit` arguments determine the training run.

In [ ]:
depth_rows = []
for max_depth in range(1, 6):
    for seed in RANDOM_SEEDS:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=TEST_SIZE, stratify=y, random_state=seed
        )
        random.seed(seed)
        custom = DecisionTreeClassifier()
        custom.fit(X_train, y_train, max_depth=max_depth)
        sklearn_tree = SklearnDecisionTreeClassifier(
            max_depth=max_depth, random_state=seed
        )
        sklearn_tree.fit(X_train, y_train)
        custom_pred = custom.predict(X_test).astype(int)
        sklearn_pred = sklearn_tree.predict(X_test).astype(int)
        depth_rows.append(
            {
                "max_depth": max_depth,
                "seed": seed,
                "custom_accuracy": accuracy_score(y_test, custom_pred),
                "sklearn_accuracy": accuracy_score(y_test, sklearn_pred),
                "custom_minus_sklearn": accuracy_score(y_test, custom_pred)
                - accuracy_score(y_test, sklearn_pred),
                "agreement": np.mean(custom_pred == sklearn_pred),
                "custom_nodes": custom.root.count_nodes(),
                "sklearn_nodes": sklearn_tree.tree_.node_count,
            }
        )

depth_results = pd.DataFrame(depth_rows)
depth_summary = (
    depth_results.groupby("max_depth")
    .agg(
        custom_accuracy_mean=("custom_accuracy", "mean"),
        sklearn_accuracy_mean=("sklearn_accuracy", "mean"),
        accuracy_difference_mean=("custom_minus_sklearn", "mean"),
        accuracy_difference_std=("custom_minus_sklearn", "std"),
        agreement_mean=("agreement", "mean"),
        agreement_min=("agreement", "min"),
        custom_nodes_mean=("custom_nodes", "mean"),
        sklearn_nodes_mean=("sklearn_nodes", "mean"),
    )
    .reset_index()
)
display(depth_summary.round(4))

## All observed disagreements and probability outputs

The table below includes every differing prediction from all 30 predeclared splits; no split is selected because it gives a preferred result. Repeated test rows are intentionally retained with their seed because the table describes repeated holdout behavior, not 900 independent observations. The probability example demonstrates a concrete API and behavioral difference. A difference in representation is an API difference; an unnormalized probability is also a correctness limitation for probabilistic use.

In [ ]:
disagreement_rows = []
for seed, values in native_predictions.items():
    mask = values["custom_pred"] != values["sklearn_pred"]
    disagreement_rows.extend(
        {
            "seed": seed,
            "iris_row": iris_row,
            "true_class": iris.target_names[true_class],
            "custom_prediction": iris.target_names[custom_class],
            "sklearn_prediction": iris.target_names[sklearn_class],
        }
        for iris_row, true_class, custom_class, sklearn_class in zip(
            values["test_ids"][mask],
            values["y_test"][mask],
            values["custom_pred"][mask],
            values["sklearn_pred"][mask],
        )
    )
all_disagreements = pd.DataFrame(disagreement_rows)
print(f"Total differing predictions across repeated holdouts: {len(all_disagreements)}")
display(all_disagreements)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=0
)
random.seed(0)
custom = DecisionTreeClassifier()
custom.fit(X_train, y_train)
sklearn_tree = SklearnDecisionTreeClassifier(random_state=0)
sklearn_tree.fit(X_train, y_train)
custom_probability = custom.predict_proba(X_test[:1])[0]
sklearn_probability = sklearn_tree.predict_proba(X_test[:1])[0]
print("Custom probability output:", custom_probability)
print("Custom probability sum:", sum(custom_probability.values()))
print("Scikit-learn classes:", sklearn_tree.classes_)
print("Scikit-learn probability row:", sklearn_probability)
print("Scikit-learn probability sum:", sklearn_probability.sum())

In [ ]:
pooled_true = np.concatenate([item["y_test"] for item in native_predictions.values()])
pooled_custom = np.concatenate([item["custom_pred"] for item in native_predictions.values()])
pooled_sklearn = np.concatenate([item["sklearn_pred"] for item in native_predictions.values()])
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, predictions, title in [
    (
        axes[0],
        pooled_custom,
        "Custom tree (pooled holdouts)",
    ),
    (
        axes[1],
        pooled_sklearn,
        "Scikit-learn (pooled holdouts)",
    ),
]:
    matrix = confusion_matrix(pooled_true, predictions, labels=range(3))
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_xticks(range(3), iris.target_names)
    ax.set_yticks(range(3), iris.target_names)
    for (row, column), value in np.ndenumerate(matrix):
        ax.text(column, row, value, ha="center", va="center")
fig.colorbar(image, ax=axes, shrink=0.8, label="count")
plt.tight_layout()
plt.show()

## Conclusions supported by this experiment

- On this small, easy, numeric dataset, the two implementations have similar average native-default accuracy and usually make the same predictions. That is a useful smoke test of the custom implementation, not proof of equivalence.
- The native-default measurements still vary by split. Individual prediction agreement is not guaranteed, and the model structures can differ even when accuracy matches.
- A shared `max_depth` makes one control comparable, but it does not remove differences in split search, stopping semantics, tie handling, validation, or probability APIs.
- This notebook does not establish production readiness, superiority, statistical significance, or performance on other datasets. A stronger evaluation would use multiple datasets, cross-validation or a predeclared external test set, and task-specific metrics.
- The most defensible description is: the project is an educational implementation whose behavior is broadly consistent with a Gini decision tree on Iris, while scikit-learn remains the more mature and feature-complete reference implementation.